# Dashboard Preparation for University Campus Health Surveillance Project 
# Author: Ari M.
# Description: Prepare a final merged dataset for Tableau by combining patient demographics with testing data and creating key fields (positivity, age group, vaccination status) for dashboard analysis.

In [1]:
import pandas as pd

In [2]:
patients = pd.read_csv("../data/cleaned/patients_cleaned.csv")
tests = pd.read_csv("../data/cleaned/tests_cleaned.csv")

print("Datasets loaded.")

Datasets loaded.


In [3]:
# Converting test_date to datetime
tests["test_date"] = pd.to_datetime(tests["test_date"])

In [4]:
# Creating positivity flag
tests["is_positive"] = tests["test_result"] == "Positive"

In [5]:
# Merging the datasets
df = tests.merge(patients, on="patient_id", how="left")

print("Datasets merged.")

Datasets merged.


### Merging Datasets

I merged test results with patient demographics so I have all the information in one dataset. 
This will make it easier for me to create visualizations in Tableau.

### Creating Datetime Fields

I’m not fully sure yet which time grouping Tableau will need, so next I’ll create a few options (monthly and quarterly) to test in the dashboard. This will allow me some flexibility when building dashboards later.

In [6]:
# Converting test_date to datetime
df["test_date"] = pd.to_datetime(df["test_date"])

df["year"] = df["test_date"].dt.year
df["month"] = df["test_date"].dt.month

# Creating year_month for Tableau time-series analysis
# Using monthly granularity for trend visualization
df["year_month"] = df["test_date"].dt.to_period("M").astype(str)

# Create quarter field for cleaner time grouping in Tableau
df["quarter"] = df["test_date"].dt.year.astype(str) + "-Q" + df["test_date"].dt.quarter.astype(str)

print("datetime fields created")

datetime fields created


In [7]:
# Creating a cleaner population field for Tableau
df["population"] = df["staff_or_student_status"]

In [8]:
# Creating age groups
bins = [17,20,25,35,50,65]
labels = ["18-20","21-25","26-35","36-50","51-65"]

df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)

print("Age groups created.")

Age groups created.


### Age Group Categorization

I created age bins to analyze positivity trends across different age groups. 
This will help me see which age groups have higher or lower infection rates.

In [9]:
# Converting vaccination field into boolean for easier filtering in Tableau
df["vaccinated"] = df["pt_has_vaccination_y_n"].map({"Yes": True, "No": False})

print(df.head())
print(df["age_group"].value_counts())

   test_id  patient_id    disease  test_date test_result report_date  \
0        1        4046  Norovirus 2019-07-18    Negative  2019-07-20   
1        2        6291  Gonorrhea 2022-07-13    Negative  2022-07-15   
2        3        6255   COVID-19 2020-01-25    Negative  2020-01-27   
3        4        1613  Chlamydia 2019-08-27    Negative  2019-08-29   
4        5        7005   COVID-19 2020-05-23    Negative  2020-05-25   

     lab_source  is_positive  age  gender  ... residence_type pt_has_test_y_n  \
0    Campus Lab        False   18    Male  ...      On-campus             Yes   
1  Hospital Lab        False   24    Male  ...      On-campus             Yes   
2    Campus Lab        False   21    Male  ...      On-campus             Yes   
3  Hospital Lab        False   23    Male  ...     Off-campus             Yes   
4  Hospital Lab        False   48  Female  ...     Off-campus             Yes   

  pt_has_vaccination_y_n  year  month  year_month  quarter population  \
0      

### Vaccination Flag

I created a new column to indicate whether each patient has been vaccinated (Yes = True, No = False). 
This will help me look at positivity by vaccination status in Tableau.

### Initial Data Inspection

- The first few rows of the dataset look correct.  
- Age group counts seem reasonable.  
- Positivity and vaccination flags appear to be applied correctly.

In [10]:
# Save final dataset for Tableau
df.to_csv("../data/cleaned/final_dataset.csv", index=False)

print("Final dataset shape:", df.shape)
print("Columns available for Tableau:", df.columns)

Final dataset shape: (25103, 21)
Columns available for Tableau: Index(['test_id', 'patient_id', 'disease', 'test_date', 'test_result',
       'report_date', 'lab_source', 'is_positive', 'age', 'gender',
       'staff_or_student_status', 'residence_type', 'pt_has_test_y_n',
       'pt_has_vaccination_y_n', 'year', 'month', 'year_month', 'quarter',
       'population', 'age_group', 'vaccinated'],
      dtype='object')


### Saving Final Dataset

I saved the final dataset as `final_dataset.csv`.  
It’s ready to be used for creating dashboards in Tableau.

This dataset is now structured in a way that should make Tableau dashboard creation more straightforward for me, especially for filtering by time, population, and disease type.